## Obiettivo: 
--> prevedere se un cliente abbandona il servizio (Churn = Yes/No) usando la regressione logistica, e interpretare i coefficienti tramite gli odds ratio.

### Il percorso ha 5 fasi:
1 → Esplorazione & Pulizia — capire il dataset, gestire valori mancanti, tipi di colonne

2 → Preprocessing — encoding delle variabili categoriche, scaling delle numeriche

3 → Training — train_test_split + LogisticRegression

4 → Valutazione — classification_report, AUC-ROC, predict_proba

5 → Odds Ratio — estrarre i coefficienti, esponenziare, interpretare

In [27]:
#import
import pandas as pd
import numpy as np

In [28]:
#import dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.xls")

In [29]:
#first exploration
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())

(7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45    

In [30]:
print(df['TotalCharges'].unique()[:20])
print(df[df['TotalCharges'] == ' '].shape)

<StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95',  '587.45',   '326.8',  '5681.1',  '5036.3',
 '2686.05', '7895.15', '1022.95', '7382.25',  '528.35',  '1862.9']
Length: 20, dtype: str
(11, 21)


## First exploration
- Row: 7043, Columns: 21
- str columns :18  | int/float columns : 3
- Nan value : none



In [31]:
#removal of customerID
df = df.drop(columns=['customerID'])
#replce and conversion of TotalCharges
df['TotalCharges'] = df['TotalCharges'].replace(' ', 0)
df['TotalCharges'] = df['TotalCharges'].astype(float)
#convert Churn
df['Churn'] = df['Churn'].map({'Yes' : 1, 'No': 0})

In [32]:
print(df.shape)
print(df['TotalCharges'].dtype)
print(df['Churn'].value_counts())

(7043, 20)
float64
Churn
0    5174
1    1869
Name: count, dtype: int64


In [33]:
#transform categorical variables in dummies (0,1)
df = pd.get_dummies (df, drop_first = True)
df = df.astype(int)

In [34]:
print(df.shape)
print(df.dtypes.value_counts())

(7043, 31)
int64    31
Name: count, dtype: int64


In [35]:
from sklearn.model_selection import train_test_split
#split x and y variables
X = df.drop(columns=["Churn"])
y = df["Churn"]

#train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [36]:
#scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#scale the continuos columns
cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [37]:
#model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(C = 1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [38]:
#prediction
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred, target_names= ["Left","Stayed"]))
print(f"AUC-ROC:{roc_auc_score(y_test, y_pred_proba):.3f}")

              precision    recall  f1-score   support

        Left       0.85      0.90      0.87      1035
      Stayed       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

AUC-ROC:0.842


In [39]:
import numpy as np
odds_ratio = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient' : model.coef_[0],
    'Odds Ratio': np.exp(model.coef_[0])
}).sort_values('Odds Ratio', ascending = False)

print(odds_ratio)



                                  Feature  Coefficient  Odds Ratio
10            InternetService_Fiber optic     1.128927    3.092336
3                            TotalCharges     0.534351    1.706341
28         PaymentMethod_Electronic check     0.385608    1.470508
26                   PaperlessBilling_Yes     0.372419    1.451240
21                        StreamingTV_Yes     0.357550    1.429822
23                    StreamingMovies_Yes     0.356488    1.428304
9                       MultipleLines_Yes     0.349907    1.418936
0                           SeniorCitizen     0.147705    1.159171
29             PaymentMethod_Mailed check     0.069722    1.072211
17                   DeviceProtection_Yes     0.024525    1.024828
5                             Partner_Yes     0.022866    1.023130
4                             gender_Male     0.022047    1.022292
27  PaymentMethod_Credit card (automatic)    -0.030927    0.969546
15                       OnlineBackup_Yes    -0.108854    0.89

In [40]:
print(odds_ratio.head())
print(odds_ratio.tail())


                           Feature  Coefficient  Odds Ratio
10     InternetService_Fiber optic     1.128927    3.092336
3                     TotalCharges     0.534351    1.706341
28  PaymentMethod_Electronic check     0.385608    1.470508
26            PaperlessBilling_Yes     0.372419    1.451240
21                 StreamingTV_Yes     0.357550    1.429822
              Feature  Coefficient  Odds Ratio
2      MonthlyCharges    -0.412963    0.661687
7    PhoneService_Yes    -0.512224    0.599162
24  Contract_One year    -0.683944    0.504623
1              tenure    -1.261580    0.283206
25  Contract_Two year    -1.317061    0.267922


## InternetService_Fiber optic → OR 3.09
The clients who have the fiber optic have 209% more probability to abbandon the service, so I advice to inspect the possibility that the service is too costly in confront of the ones on the market or the service is not working properly.

## Two years contract versus month to month contract
The clients who have been using the service for longer and have two years contract are more inclined to stay with the company compared to the ones with the month to month contract or that are with the company less probably beacuse they are more loyal and don't feel the need to change so we should invest in making the clients with the month to month contract and the new clients to stay with personalized offers.

## Electronic Payment
The clients who use the eletronic payment method are more inclined to leave the service, probably because they are more techlogically educated they can compare this offer with others company ones and decide to leave if there are better opportunities. so i advice to offer a promotional price to the ones who use electronic payments.

# Telco Customer Churn — Business Insights
### Logistic Regression Analysis | Develhope Data Science Program

---

## Model Performance Summary

| Metric | Value |
|---|---|
| Overall Accuracy | 81% |
| AUC-ROC | 0.842 |
| Recall (Churn class) | 56% |
| Precision (Churn class) | 66% |

> ⚠️ **Note for stakeholders:** The model correctly identifies 56% of customers who will churn. Improving recall is the primary optimization target, as missed churners (false negatives) represent direct revenue loss.

---

## Key Drivers of Churn — Odds Ratio Analysis

| Feature | Odds Ratio | Effect |
|---|---|---|
| InternetService — Fiber Optic | 3.09 | ↑ Increases churn |
| TotalCharges | 1.71 | ↑ Increases churn |
| PaymentMethod — Electronic Check | 1.47 | ↑ Increases churn |
| PaperlessBilling | 1.45 | ↑ Increases churn |
| Contract — One Year | 0.50 | ↓ Reduces churn |
| Tenure | 0.28 | ↓ Reduces churn |
| Contract — Two Year | 0.27 | ↓ Reduces churn |

> **Reference category for Contract:** Month-to-month (implicit baseline).

---

## Business Recommendations

### 1. 🔴 Fiber Optic Internet Service (OR: 3.09)
Customers subscribed to the fiber optic internet service are **209% more likely to churn** compared to those without it. This counterintuitive finding suggests potential issues with service quality or pricing competitiveness. 

**Recommended action:** Conduct a targeted satisfaction survey among fiber optic users and benchmark pricing against key market competitors.

---

### 2. 🟢 Contract Type & Tenure (OR: 0.27 / 0.28)
Customers on two-year contracts and those with longer tenure are significantly less likely to churn — up to **73% less** compared to month-to-month subscribers. Long-term commitment is the strongest retention driver identified by the model.

**Recommended action:** Introduce incentives that encourage month-to-month customers and new subscribers to upgrade to longer-term contracts, such as discounted rates or loyalty rewards.

---

### 3. 🟡 Electronic Check Payment Method (OR: 1.47)
Customers paying via electronic check are **47% more likely to abandon** the service. This group tends to be more digitally engaged and therefore more likely to actively compare competitor offerings online.

**Recommended action:** Target this segment with personalized retention offers or switching incentives before they initiate cancellation.

---

## Methodology Notes

- **Dataset:** IBM Telco Customer Churn (7,043 records, 20 features after cleaning)
- **Model:** Logistic Regression (`max_iter=1000`, `C=1.0`)
- **Preprocessing:** One-Hot Encoding (`drop_first=True`), StandardScaler on continuous features only (`tenure`, `MonthlyCharges`, `TotalCharges`)
- **Split:** 80/20 train/test with stratification on target variable
- **Odds Ratio:** Computed as `e^coefficient` for each feature

---

*Analysis conducted as part of the Develhope Data Science & Behavioral Analysis Program — Exercise 3.*

In [ ]:
#struttura per i prossimi progetti:

#Tabella performance in cima 
# — il manager vede subito i numeri chiave
#Tabella odds ratio ordinata 
# — immediata da leggere, con frecce ↑↓ per direzione dell'effetto
#3 sezioni recommendation con emoji colore (🔴 rischio, 🟢 opportunità, 🟡 attenzione) 
# — gerarchizza visivamente la priorità
#Methodology Notes in fondo 
# — trasparenza tecnica per chi vuole approfondire, ma non disturba la lettura business
#Ogni raccomandazione segue sempre lo stesso schema: 
# dato → interpretazione → azione#